# CatBoost Model Results Analysis

Training is implemented in `prs_its.training` and run with `prs-its-train`. This notebook only analyzes the saved, leakage-safe training artifacts.

## Load Results

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from prs_its.calibration import calibration_curve_frame, prediction_distribution
from prs_its.metrics import audit_metrics, evaluate_probabilities
from prs_its.submission import prediction_summary

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate pyproject.toml.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
METRICS_DIR = OUTPUT_DIR / 'metrics'
MODELS_DIR = OUTPUT_DIR / 'models'
OOF_DIR = OUTPUT_DIR / 'oof'
SUBMISSIONS_DIR = OUTPUT_DIR / 'submissions'

required_paths = [
    METRICS_DIR / 'catboost_experiments.csv',
    METRICS_DIR / 'catboost_fold_metrics.csv',
    METRICS_DIR / 'catboost_fairness.csv',
    MODELS_DIR / 'catboost_final_config.json',
    OOF_DIR / 'catboost_oof.csv',
    SUBMISSIONS_DIR / 'catboost_submission.csv',
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing training artifacts: {missing}. Run prs-its-train first.')

experiments = pd.read_csv(METRICS_DIR / 'catboost_experiments.csv')
fold_metrics = pd.read_csv(METRICS_DIR / 'catboost_fold_metrics.csv')
fairness = pd.read_csv(METRICS_DIR / 'catboost_fairness.csv')
oof = pd.read_csv(OOF_DIR / 'catboost_oof.csv')
submission = pd.read_csv(SUBMISSIONS_DIR / 'catboost_submission.csv')
with (MODELS_DIR / 'catboost_final_config.json').open() as file:
    final_config = json.load(file)

## Experiment Selection

In [ ]:
display(experiments.sort_values(['normalized_recall_at_5pct', 'average_precision'], ascending=False))
display(pd.Series(final_config))

## Fold Stability and Audit Allocation

In [ ]:
display(fold_metrics)
display(fold_metrics[['average_precision', 'brier_score', 'normalized_recall_at_5pct', 'precision_at_5pct', 'lift_at_5pct', 'best_iteration']].agg(['mean', 'std', 'min', 'max']))
final_metrics = evaluate_probabilities(oof['label'], oof['fraud_probability_final'])
audit_table = pd.DataFrame([audit_metrics(oof['label'], oof['fraud_probability_final'], fraction) for fraction in (0.03, 0.05, 0.07)])
display(pd.Series(final_metrics))
display(audit_table)

## Calibration and Prediction Distribution

In [ ]:
calibration = calibration_curve_frame(oof['label'], oof['fraud_probability_final'])
display(calibration)
display(prediction_distribution(oof['label'], oof['fraud_probability_final']))
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='ideal')
plt.plot(calibration['mean_predicted_probability'], calibration['observed_fraud_rate'], marker='o', label=final_config['calibration'])
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed fraud rate')
plt.legend()
plt.title('OOF Reliability Curve')
plt.show()

## Explainability

In [ ]:
for filename in ['catboost_feature_importance.csv', 'catboost_shap_summary.csv', 'catboost_representative_explanations.csv']:
    path = METRICS_DIR / filename
    if path.exists():
        display(pd.read_csv(path).head(25))
    else:
        print(f'{filename} will be created by the next prs-its-train run.')
print('Feature importance and SHAP values describe model behavior and predictive association, not causality.')

## Policyholder Protection

In [ ]:
display(fairness.query('audit_fraction == 0.05'))
print('Unnecessary-audit rates are restricted to legitimate claims. Raw jkpst categories are not assigned demographic meanings without the data dictionary.')

## Submission Validation

In [ ]:
assert submission.columns.tolist() == ['claim_id', 'fraud_probability']
assert submission['fraud_probability'].notna().all()
assert np.isfinite(submission['fraud_probability']).all()
assert submission['fraud_probability'].between(0, 1).all()
display(prediction_summary(submission['fraud_probability']))
display(submission.head())

## Limitations

The results depend on training-data labels and feature availability. The official data dictionary is still needed to confirm feature semantics and decision-time availability. Repeated feature profiles, distribution shift, calibration drift, fairness exposure, and private-test uncertainty require ongoing monitoring.